# Azure LLM Inference Trace — Statistics

Input/Output token length statistics and request arrival rate analysis.

All logic is in reusable functions — call them with different parameters to compare configurations easily.

In [1]:
import csv
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from pathlib import Path
from IPython.display import display, Markdown

## Functions

In [2]:
def load_trace(csv_path: str, max_requests: int = None) -> pd.DataFrame:
    """
    Load Azure trace CSV into a DataFrame.

    Returns DataFrame with columns:
        timestamp, input_tokens, output_tokens, total_tokens, arrival_sec
    where arrival_sec is relative seconds from the first request.
    """
    rows = []
    with open(csv_path, 'r') as f:
        reader = csv.DictReader(f)
        for i, row in enumerate(reader):
            if max_requests is not None and i >= max_requests:
                break
            rows.append({
                'timestamp': datetime.fromisoformat(row['TIMESTAMP']),
                'input_tokens': int(row['ContextTokens']),
                'output_tokens': int(row['GeneratedTokens']),
            })

    df = pd.DataFrame(rows)
    df['total_tokens'] = df['input_tokens'] + df['output_tokens']
    df['arrival_sec'] = (df['timestamp'] - df['timestamp'].iloc[0]).dt.total_seconds()

    print(f"Loaded {len(df):,} requests from {Path(csv_path).name}")
    print(f"Time span: {df['timestamp'].iloc[0]} → {df['timestamp'].iloc[-1]}")
    print(f"Duration: {df['arrival_sec'].iloc[-1]:.1f}s ({df['arrival_sec'].iloc[-1]/60:.1f} min)")
    return df

In [3]:
def filter_window(df: pd.DataFrame, start_min: float = 0, end_min: float = None) -> pd.DataFrame:
    """
    Filter trace to a relative time window.

    Args:
        df: DataFrame from load_trace()
        start_min: Start of window in minutes (relative to first request)
        end_min: End of window in minutes (None = end of trace)

    Returns:
        Filtered DataFrame with arrival_sec recalculated from window start.
    """
    first_ts = df['timestamp'].iloc[0]
    t_start = first_ts + timedelta(minutes=start_min)

    if end_min is not None:
        t_end = first_ts + timedelta(minutes=end_min)
        mask = (df['timestamp'] >= t_start) & (df['timestamp'] < t_end)
        label = f"{start_min}–{end_min} min"
    else:
        mask = df['timestamp'] >= t_start
        label = f"{start_min} min–end"

    filtered = df[mask].copy()
    if len(filtered) > 0:
        filtered['arrival_sec'] = (filtered['timestamp'] - filtered['timestamp'].iloc[0]).dt.total_seconds()

    print(f"Window [{label}]: {len(filtered):,} requests")
    return filtered

In [4]:
def _percentile_row(arr, name):
    """Build a statistics row for a numeric array."""
    return {
        "Metric": name,
        "Count": len(arr),
        "Mean": f"{np.mean(arr):.2f}",
        "Std": f"{np.std(arr):.2f}",
        "Min": int(np.min(arr)),
        "P10": f"{np.percentile(arr, 10):.0f}",
        "P25": f"{np.percentile(arr, 25):.0f}",
        "Median": f"{np.median(arr):.0f}",
        "P75": f"{np.percentile(arr, 75):.0f}",
        "P90": f"{np.percentile(arr, 90):.0f}",
        "P99": f"{np.percentile(arr, 99):.0f}",
        "Max": int(np.max(arr)),
    }


def token_stats(df: pd.DataFrame, title: str = None):
    """
    Display input/output/total token length statistics.

    Args:
        df: DataFrame from load_trace() or filter_window()
        title: Optional title override
    """
    if len(df) == 0:
        print("No data.")
        return

    table = pd.DataFrame([
        _percentile_row(df['input_tokens'].values, 'Input tokens'),
        _percentile_row(df['output_tokens'].values, 'Output tokens'),
        _percentile_row(df['total_tokens'].values, 'Total tokens (in+out)'),
    ])

    if title:
        display(Markdown(f"### {title}"))
    else:
        display(Markdown("### Token Length Statistics"))
    display(table.set_index('Metric'))

In [5]:
def arrival_stats(df: pd.DataFrame, title: str = None):
    """
    Display request arrival rate and inter-arrival time statistics.

    Args:
        df: DataFrame from load_trace() or filter_window()
        title: Optional title override
    """
    if len(df) < 2:
        print("Not enough data.")
        return

    arr = df['arrival_sec'].values
    max_sec = int(arr[-1]) + 1

    # Per-second counts
    per_sec = np.zeros(max_sec, dtype=int)
    for t in arr:
        per_sec[min(int(t), max_sec - 1)] += 1

    # Per-minute counts
    max_min = int(arr[-1] / 60) + 1
    per_min = np.zeros(max_min, dtype=int)
    for t in arr:
        per_min[min(int(t / 60), max_min - 1)] += 1

    # Inter-arrival times
    iat_ms = np.diff(arr) * 1000

    rows = [
        _percentile_row(per_sec, 'Arrival rate (req/s)'),
        _percentile_row(per_min, 'Arrival rate (req/min)'),
    ]

    if title:
        display(Markdown(f"### {title}"))
    else:
        display(Markdown("### Arrival Rate Statistics"))
    display(pd.DataFrame(rows).set_index('Metric'))

    if len(iat_ms) > 0:
        display(Markdown("### Inter-Arrival Time"))
        display(pd.DataFrame([_percentile_row(iat_ms, 'Inter-arrival time (ms)')]).set_index('Metric'))

In [6]:
def summary(df: pd.DataFrame, label: str = "Full trace"):
    """
    Display a one-row summary of the trace/window.

    Args:
        df: DataFrame from load_trace() or filter_window()
        label: Description label for the window
    """
    if len(df) < 2:
        print("Not enough data.")
        return

    dur = df['arrival_sec'].iloc[-1]
    s = {
        "Window": label,
        "Requests": f"{len(df):,}",
        "Duration (sec)": f"{dur:.1f}",
        "Duration (min)": f"{dur/60:.1f}",
        "Avg req/s": f"{len(df)/dur:.2f}" if dur > 0 else "N/A",
        "Total input tokens": f"{df['input_tokens'].sum():,}",
        "Total output tokens": f"{df['output_tokens'].sum():,}",
        "Avg input len": f"{df['input_tokens'].mean():.1f}",
        "Avg output len": f"{df['output_tokens'].mean():.1f}",
        "Median input len": f"{df['input_tokens'].median():.0f}",
        "Median output len": f"{df['output_tokens'].median():.0f}",
    }

    display(Markdown(f"### Summary — {label}"))
    display(pd.DataFrame([s]).T.rename(columns={0: 'Value'}))

In [7]:
def compare_windows(df: pd.DataFrame, windows: list):
    """
    Compare statistics across multiple time windows.

    Args:
        df: DataFrame from load_trace()
        windows: List of (start_min, end_min) tuples.
                 end_min=None means end of trace.

    Example:
        compare_windows(df, [(0, 15), (15, 30), (0, None)])
    """
    first_ts = df['timestamp'].iloc[0]
    rows = []

    for w_start, w_end in windows:
        t_start = first_ts + timedelta(minutes=w_start)
        if w_end is not None:
            t_end = first_ts + timedelta(minutes=w_end)
            mask = (df['timestamp'] >= t_start) & (df['timestamp'] < t_end)
            label = f"{w_start}–{w_end} min"
        else:
            mask = df['timestamp'] >= t_start
            label = f"{w_start} min–end"

        w = df[mask]
        if len(w) == 0:
            continue

        w_arr = (w['timestamp'] - w['timestamp'].iloc[0]).dt.total_seconds().values
        w_dur = w_arr[-1] if len(w_arr) > 1 else 1

        rows.append({
            'Window': label,
            'Requests': len(w),
            'Avg req/s': f"{len(w)/w_dur:.2f}" if w_dur > 0 else 'N/A',
            'Avg input': f"{w['input_tokens'].mean():.0f}",
            'Med input': f"{w['input_tokens'].median():.0f}",
            'Avg output': f"{w['output_tokens'].mean():.0f}",
            'Med output': f"{w['output_tokens'].median():.0f}",
            'P90 input': f"{np.percentile(w['input_tokens'], 90):.0f}",
            'P90 output': f"{np.percentile(w['output_tokens'], 90):.0f}",
            'P99 input': f"{np.percentile(w['input_tokens'], 99):.0f}",
            'P99 output': f"{np.percentile(w['output_tokens'], 99):.0f}",
        })

    if rows:
        display(Markdown('### Time Window Comparison'))
        display(pd.DataFrame(rows).set_index('Window'))

---
## Usage Examples

In [8]:
# Load trace
df = load_trace("AzureLLMInferenceConvTrace_pruned_2048.csv")

Loaded 16,663 requests from AzureLLMInferenceConvTrace_pruned_2048.csv
Time span: 2023-11-16 18:15:46.680590 → 2023-11-16 19:14:08.402527
Duration: 3501.7s (58.4 min)


In [9]:
# Full trace statistics
token_stats(df)
arrival_stats(df)
summary(df)

### Token Length Statistics

,Count,Mean,Std,Min,P10,P25,Median,P75,P90,P99,Max
Metric,,,,,,,,,,,
Input tokens,16663,762.80,444.71,2,203,391,947,1094,1273,1911,2047
Output tokens,16663,232.40,164.77,7,68,92,157,397,429,608,1000
Total tokens (in+out),16663,995.20,547.56,64,352,480,1129,1492,1586,2028,2488


### Arrival Rate Statistics

,Count,Mean,Std,Min,P10,P25,Median,P75,P90,P99,Max
Metric,,,,,,,,,,,
Arrival rate (req/s),3502,4.76,2.45,0,2,3,5,6,8,11,15
Arrival rate (req/min),59,282.42,58.46,36,229,250,282,314,351,402,427


### Inter-Arrival Time

,Count,Mean,Std,Min,P10,P25,Median,P75,P90,P99,Max
Metric,,,,,,,,,,,
Inter-arrival time (ms),16662,210.16,226.04,0,19,58,139,287,490,1018,4314


### Summary — Full trace

,Value
Window,Full trace
Requests,"16,663"
Duration (sec),3501.7
Duration (min),58.4
Avg req/s,4.76
Total input tokens,"12,710,610"
Total output tokens,"3,872,466"
Avg input len,762.8
Avg output len,232.4
Median input len,947


In [10]:
# Specific time window: first 15 minutes
w = filter_window(df, 0, 15)
token_stats(w, title="Token Stats (0–15 min)")
arrival_stats(w, title="Arrival Rate (0–15 min)")
summary(w, label="0–15 min")

Window [0–15 min]: 3,905 requests


### Token Stats (0–15 min)

,Count,Mean,Std,Min,P10,P25,Median,P75,P90,P99,Max
Metric,,,,,,,,,,,
Input tokens,3905,853.80,413.56,2,212,401,1027,1115,1231,1896,2045
Output tokens,3905,278.06,169.70,10,70,99,377,410,436,638,1000
Total tokens (in+out),3905,1131.86,523.76,95,396,498,1426,1517,1602,2007,2236


### Arrival Rate (0–15 min)

,Count,Mean,Std,Min,P10,P25,Median,P75,P90,P99,Max
Metric,,,,,,,,,,,
Arrival rate (req/s),900,4.34,2.20,0,2,3,4,6,7,10,12
Arrival rate (req/min),15,260.33,30.76,176,237,250,254,282,299,300,300


### Inter-Arrival Time

,Count,Mean,Std,Min,P10,P25,Median,P75,P90,P99,Max
Metric,,,,,,,,,,,
Inter-arrival time (ms),3904,230.50,245.40,0,21,66,157,319,539,1077,4314


### Summary — 0–15 min

,Value
Window,0–15 min
Requests,"3,905"
Duration (sec),899.9
Duration (min),15.0
Avg req/s,4.34
Total input tokens,"3,334,089"
Total output tokens,"1,085,819"
Avg input len,853.8
Avg output len,278.1
Median input len,1027


In [11]:
# Compare multiple windows side by side
compare_windows(df, [
    (0, 15),
    (15, 30),
    (30, 60),
    (60, 120),
    (0, None),  # full trace
])

### Time Window Comparison

,Requests,Avg req/s,Avg input,Med input,Avg output,Med output,P90 input,P90 output,P99 input,P99 output
Window,,,,,,,,,,
0–15 min,3905,4.34,854,1027,278,377,1231,436,1896,638
15–30 min,4598,5.11,715,424,216,133,1223,427,1890,606
30–60 min,8160,4.80,746,845,220,140,1312,425,1932,592
0 min–end,16663,4.76,763,947,232,157,1273,429,1911,608


In [12]:
# Specific time window: first 10 minutes
w = filter_window(df, 0, 10)
token_stats(w, title="Token Stats (0–10 min)")
arrival_stats(w, title="Arrival Rate (0–10 min)")
summary(w, label="0–10 min")

Window [0–10 min]: 2,564 requests


### Token Stats (0–10 min)

,Count,Mean,Std,Min,P10,P25,Median,P75,P90,P99,Max
Metric,,,,,,,,,,,
Input tokens,2564,852.21,412.55,2,209,401,1025,1120,1231,1866,2042
Output tokens,2564,282.26,168.04,10,70,102,379,411,440,623,1000
Total tokens (in+out),2564,1134.47,524.99,95,388,499,1424,1518,1607,1988,2236


### Arrival Rate (0–10 min)

,Count,Mean,Std,Min,P10,P25,Median,P75,P90,P99,Max
Metric,,,,,,,,,,,
Arrival rate (req/s),600,4.27,2.23,0,2,3,4,6,8,10,12
Arrival rate (req/min),10,256.40,35.22,176,227,244,250,282,300,300,300


### Inter-Arrival Time

,Count,Mean,Std,Min,P10,P25,Median,P75,P90,P99,Max
Metric,,,,,,,,,,,
Inter-arrival time (ms),2563,234.09,257.80,0,23,66,156,320,560,1080,4314


### Summary — 0–10 min

,Value
Window,0–10 min
Requests,"2,564"
Duration (sec),600.0
Duration (min),10.0
Avg req/s,4.27
Total input tokens,"2,185,073"
Total output tokens,"723,719"
Avg input len,852.2
Avg output len,282.3
Median input len,1025


In [13]:
# Specific time window: first 10 minutes
w = filter_window(df, 0, 3)
token_stats(w, title="Token Stats (0–3 min)")
arrival_stats(w, title="Arrival Rate (0–3 min)")
summary(w, label="0–3 min")

Window [0–3 min]: 727 requests


### Token Stats (0–3 min)

,Count,Mean,Std,Min,P10,P25,Median,P75,P90,P99,Max
Metric,,,,,,,,,,,
Input tokens,727,739.41,426.82,2,181,374,972,1082,1163,1372,2042
Output tokens,727,274.87,160.25,12,78,112,365,410,429,594,1000
Total tokens (in+out),727,1014.29,550.57,95,329,436,1365,1490,1566,1812,2113


### Arrival Rate (0–3 min)

,Count,Mean,Std,Min,P10,P25,Median,P75,P90,P99,Max
Metric,,,,,,,,,,,
Arrival rate (req/s),180,4.04,2.35,0,1,2,4,5,8,9,10
Arrival rate (req/min),3,242.33,50.99,176,191,214,251,276,290,299,300


### Inter-Arrival Time

,Count,Mean,Std,Min,P10,P25,Median,P75,P90,P99,Max
Metric,,,,,,,,,,,
Inter-arrival time (ms),726,247.79,313.51,0,21,64,170,326,576,1232,4314


### Summary — 0–3 min

,Value
Window,0–3 min
Requests,727
Duration (sec),179.9
Duration (min),3.0
Avg req/s,4.04
Total input tokens,"537,553"
Total output tokens,"199,834"
Avg input len,739.4
Avg output len,274.9
Median input len,972


In [14]:
# Specific time window: first 10 minutes
w = filter_window(df, 0, 10)
token_stats(w, title="Token Stats (0–10 min)")
arrival_stats(w, title="Arrival Rate (0–10 min)")
summary(w, label="0–10 min")

Window [0–10 min]: 2,564 requests


### Token Stats (0–10 min)

,Count,Mean,Std,Min,P10,P25,Median,P75,P90,P99,Max
Metric,,,,,,,,,,,
Input tokens,2564,852.21,412.55,2,209,401,1025,1120,1231,1866,2042
Output tokens,2564,282.26,168.04,10,70,102,379,411,440,623,1000
Total tokens (in+out),2564,1134.47,524.99,95,388,499,1424,1518,1607,1988,2236


### Arrival Rate (0–10 min)

,Count,Mean,Std,Min,P10,P25,Median,P75,P90,P99,Max
Metric,,,,,,,,,,,
Arrival rate (req/s),600,4.27,2.23,0,2,3,4,6,8,10,12
Arrival rate (req/min),10,256.40,35.22,176,227,244,250,282,300,300,300


### Inter-Arrival Time

,Count,Mean,Std,Min,P10,P25,Median,P75,P90,P99,Max
Metric,,,,,,,,,,,
Inter-arrival time (ms),2563,234.09,257.80,0,23,66,156,320,560,1080,4314


### Summary — 0–10 min

,Value
Window,0–10 min
Requests,"2,564"
Duration (sec),600.0
Duration (min),10.0
Avg req/s,4.27
Total input tokens,"2,185,073"
Total output tokens,"723,719"
Avg input len,852.2
Avg output len,282.3
Median input len,1025
